In [1]:
import pandas as pd
from datetime import datetime
from bs4 import BeautifulSoup
import requests
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
import time
import datetime
import re
from unidecode import unidecode
pd.options.display.max_columns = 0
pd.options.display.max_rows = 30
pd.options.display.max_colwidth = 200

In [2]:
today = datetime.date.today()

In [3]:
def get_todays_teams():
    if len(str(today.month)) == 2:
        mon = str(today.month)
    else:
        mon = '0'+str(today.month)

    if len(str(today.day)) == 2:
        day = str(today.day)
    else:
        day = '0'+str(today.day)
    url = 'https://api3.natst.at/6843-c01eff/games/NBA/' + str(today.year) + '-' + mon + '-' + day
    games = requests.get(url).json()
    teams = []
    for key,value in games['games'].items():
        teams.append(value['visitor-code'])
        teams.append(value['home-code']) 
    return teams

In [4]:
if len(str(today.month)) == 2:
    mon = str(today.month)
else:
    mon = '0'+str(today.month)

if len(str(today.day)) == 2:
    day = str(today.day)
else:
    day = '0'+str(today.day)
url = 'https://api3.natst.at/6843-c01eff/games/NBA/' + str(today.year) + '-' + mon + '-' + day
games = requests.get(url).json()
teams = []

In [5]:
todays_teams = get_todays_teams()

In [6]:
def get_todays_games():
    if len(str(today.month)) == 2:
        mon = str(today.month)
    else:
        mon = '0'+str(today.month)

    if len(str(today.day)) == 2:
        day = str(today.day)
    else:
        day = '0'+str(today.day)
    url = 'https://api3.natst.at/6843-c01eff/games/NBA/' + str(today.year) + '-' + mon + '-' + day
    games = requests.get(url).json()
    teams = {}
    for key,value in games['games'].items():
        teams[value['visitor-code']] = value['home-code']
        teams[value['home-code']] = value['visitor-code']
    return teams

In [7]:
todays_games = get_todays_games()

In [10]:
def get_teams():
    url = 'https://api3.natst.at/6843-c01eff/teams/NBA/2026'
    output_teams = []
    teams = requests.get(url).json()
    for key,value in teams['teams'].items():
        output_teams.append({'name':value['name'],'team-code':value['code'],'key':key})
    return pd.DataFrame(output_teams)
        

In [11]:
def get_players(team_code):
    url = 'https://api3.natst.at/6843-c01eff/players/NBA/' + team_code
    output_players = []
    players = requests.get(url).json()
    for cp, player in players['players'].items():
        output_players.append(player)
    return pd.DataFrame(output_players)
        

In [12]:
def get_player_stats(code):
    try:
        url = 'https://api3.natst.at/6843-c01eff/players/NBA/' + str(code)
        print(url)
        stats = requests.get(url).json()
        output = []
        s_tag = list(stats['players'][list(stats['players'].keys())[0]]['seasons']['season_2026'].keys())[1]
        for key,game in stats['players'][list(stats['players'].keys())[0]]['seasons']['season_2026'][s_tag]['playerperfs'].items():
            match_minutes = re.search(r'(\d+)m', game['statline'])
            if match_minutes:
                res1 = match_minutes.group(1)
                game['minutes'] = res1
            else:
                game['minutes'] = 0
            match_points = re.search(r'(\d+)p', game['statline'])
            if match_points:
                res2 = match_points.group(1)
                game['points'] = res2
            else:
                game['points'] = 0
            match_reb = re.search(r'(\d+)r', game['statline'])
            if match_reb:
                res3 = match_reb.group(1)
                game['rebounds'] = res3
            else:
                game['rebounds'] = 0
            match_assists = re.search(r'(\d+)a', game['statline'])
            if match_assists:
                res4 = match_assists.group(1)
                game['assists'] = res4
            else:
                game['assists'] = 0
            game['code'] = code
            output.append(game)
        return pd.DataFrame(output)
    except:
        print("Request Failed!")
        print(url)
        return pd.DataFrame()

In [13]:
def get_player_positions():
    positions = []
    response = requests.get('https://www.basketball-reference.com/leagues/NBA_2026_play-by-play.html')
    html = response.text
    soup = BeautifulSoup(html, "lxml")
    table = soup.find_all('table', id='pbp_stats')
    rows = table[0].find_all('tr')
    for row in rows:
        cells = row.find_all('td')
        if len(cells) > 0:
            positions.append({'name':cells[0].text,'position':cells[3].text})
    return pd.DataFrame(positions)

In [14]:
player_positions = get_player_positions()

In [15]:
def get_game_links():
    options = webdriver.ChromeOptions()
    #options.add_argument("--enable-javascript")
    options.add_argument("javascript.enabled")
    driver = webdriver.Chrome(options=options)
    driver.get("https://www.bovada.lv/sports/basketball/nba")
    time.sleep(5)
    #games = driver.find_element(By.CLASS_NAME,"game-view-cta")
    html = driver.page_source
    driver.quit()
    soup = BeautifulSoup(html)
    todays_game_links = []
    for tag in soup.findAll('a',attrs={'class':'game-view-cta'}):
        todays_game_links.append('https://www.bovada.lv'+tag.get('href'))
    return todays_game_links

In [16]:
def get_player_props(link):
    props = []
    options = webdriver.ChromeOptions()
    #options.add_argument("--enable-javascript")
    options.add_argument("javascript.enabled")
    driver = webdriver.Chrome(options=options)
    driver.get(link)
    time.sleep(10)
    #games = driver.find_element(By.CLASS_NAME,"game-view-cta")
    html = driver.page_source
    driver.quit()
    soup = BeautifulSoup(html)
    for tag in soup.findAll('sp-single-market'):
        header = tag.find('h3')
        pattern = r"(.+)\s+-\s+(.+)\s+\((\S{3})\)"
        matches = re.findall(pattern, header.text)
        if matches:
            spread = tag.find('ul', class_='spread-header')
            if spread:
                props.append({'type':matches[0][0].strip(),'player':matches[0][1],'team':matches[0][2],'spread':spread.text})
    return pd.DataFrame(props)

In [17]:
def get_defense_by_position():
    response = requests.get("https://hashtagbasketball.com/nba-defense-vs-position")
    html = response.text
    
    # Create the soup object
    soup = BeautifulSoup(html, "lxml")
    table = soup.find_all('table', class_='table table-sm table-bordered table-striped table--statistics')
    rows = table[2].find_all('tr')

    defense_data = []
    headers=['','','points','fg%','ft%','3pm','rebounds','assists','steals','blocks','turnovers']
    for row in rows:
        cells = row.find_all('td')
        if len(cells) > 0:
            for i in range(2,10):
                try:
                    defense_data.append({'position':cells[0].text,'team':cells[1].text.split()[0],'team_rank':cells[1].text.split()[1],'stat':headers[i],'value':cells[i].text.split()[0],'rank':cells[i].text.split()[1]})
                except:
                    print(i)
    df = pd.DataFrame(defense_data)
    df['value'] = df['value'].astype(float)
    df['rank'] = df['rank'].astype(int)
    df['team_rank'] = df['team_rank'].astype(int)
    return pd.DataFrame(defense_data)

In [18]:
defense = get_defense_by_position()

In [19]:
defense

,position,team,team_rank,stat,value,rank
0,PF,OKC,1,points,19.7,13
1,PF,OKC,1,fg%,43.3,34
2,PF,OKC,1,ft%,61.6,4
3,PF,OKC,1,3pm,1.8,31
4,PF,OKC,1,rebounds,9.5,99
...,...,...,...,...,...,...
1195,C,LAC,150,3pm,1.7,25
1196,C,LAC,150,rebounds,13.1,123
1197,C,LAC,150,assists,4.1,44
1198,C,LAC,150,steals,1.9,109


In [20]:
defense = defense.sort_values(by='value', ascending=True)

# Step 2: Count occurrences of unique pairs in 'col1' and 'col2'
defense['rank'] = defense.groupby(['position', 'stat'])['value'].rank(ascending=True, method='min')

In [21]:
defense['rank'] = defense['rank'].astype(int)

In [22]:
teams = get_teams()

In [23]:
players = pd.DataFrame()
for i in list(teams['team-code']):
    players = pd.concat([players,get_players(i)])

In [24]:
stats = pd.DataFrame()
todays_players = players[players['team-code'].isin(todays_teams)]
print(todays_players.shape)
for code in list(players['code'].unique()):
    stats = pd.concat([stats,get_player_stats(code)])
stats['minutes'] = stats['minutes'].fillna(0)
stats['points'] = stats['points'].fillna(0)
stats['rebounds'] = stats['rebounds'].fillna(0)
stats['assists'] = stats['assists'].fillna(0)

(277, 4)
https://api3.natst.at/6843-c01eff/players/NBA/1218683
https://api3.natst.at/6843-c01eff/players/NBA/47267142
https://api3.natst.at/6843-c01eff/players/NBA/58075832
https://api3.natst.at/6843-c01eff/players/NBA/52566257
https://api3.natst.at/6843-c01eff/players/NBA/47267156
https://api3.natst.at/6843-c01eff/players/NBA/40722898
https://api3.natst.at/6843-c01eff/players/NBA/329830
https://api3.natst.at/6843-c01eff/players/NBA/52851533
https://api3.natst.at/6843-c01eff/players/NBA/89072740
https://api3.natst.at/6843-c01eff/players/NBA/10003064
https://api3.natst.at/6843-c01eff/players/NBA/3103
https://api3.natst.at/6843-c01eff/players/NBA/58075633
https://api3.natst.at/6843-c01eff/players/NBA/52566303
https://api3.natst.at/6843-c01eff/players/NBA/58075635
https://api3.natst.at/6843-c01eff/players/NBA/645263
https://api3.natst.at/6843-c01eff/players/NBA/538985
https://api3.natst.at/6843-c01eff/players/NBA/119722
https://api3.natst.at/6843-c01eff/players/NBA/40722920
https://api3.n

In [25]:
stats.head()

,id,game-code,gameday,opponent,opponent-team-code,winorloss,result,statline,minutes,points,rebounds,assists,code
0,12449415,1437601,2025-10-22,Toronto Raptors,TOR,L,"L, 118-138",28m 10p 4r 4a 1b,28,10,4,4,1218683
1,12460973,1437612,2025-10-24,Orlando Magic,ORL,W,"W, 111-107",31m 19p 1r 4a,31,19,1,4,1218683
2,12461312,1437625,2025-10-25,Oklahoma City Thunder,OKC,L,"L, 100-117",30m 17p 3r 2a 1b,30,17,3,2,1218683
3,12461726,1437640,2025-10-27,Chicago Bulls,CHI,L,"L, 123-128",30m 17p 3r 3a,30,17,3,3,1218683
4,12462152,1438737,2025-10-29,Brooklyn Nets,BRK,W,"W, 117-112",32m 18p 3r 3a 3b,32,18,3,3,1218683


In [26]:
players['name'] = players['name'].apply(lambda x: unidecode(x))

In [27]:
players.shape

(461, 4)

In [28]:
player_positions.shape

(461, 2)

In [29]:
players.merge(player_positions,how='left').shape

(461, 5)

In [30]:
player_positions['name'] = player_positions['name'].str.replace(' Jr.','')
player_positions['name'] = player_positions['name'].str.replace(' Sr.','')
player_positions['name'] = player_positions['name'].str.replace('Herbert Jones','Herb Jones')
player_positions['name'] = player_positions['name'].str.replace('Vince Williams','Vincent Williams')
player_positions['name'] = player_positions['name'].str.replace('Nicolas Claxton','Nic Claxton')
player_positions['name'] = player_positions['name'].str.replace('Bogdan BogdanoviÄ\x87','Bogdan Bogdanovic')
player_positions['name'] = player_positions['name'].str.replace('Kristaps PorziÅ\x86Ä£is','Kristaps Porzingis')
player_positions['name'] = player_positions['name'].str.replace('Cui Yongxi','Yongxi Cui')
player_positions['name'] = player_positions['name'].str.replace('Dennis SchrÃ¶der','Dennis Schroder')
player_positions['name'] = player_positions['name'].str.replace('Moussa DiabatÃ©','Moussa Diabate')
player_positions['name'] = player_positions['name'].str.replace('Vasilije MiciÄ\x87','Vasilije Micic')
player_positions['name'] = player_positions['name'].str.replace('Tidjane SalaÃ¼n','Tidjane Salaun')
player_positions['name'] = player_positions['name'].str.replace('E.J. Liddell','EJ Liddell')
player_positions['name'] = player_positions['name'].str.replace('Nikola VuÄ\x8deviÄ\x87','Nikola Vucevic')
player_positions['name'] = player_positions['name'].str.replace('Luka DonÄ\x8diÄ\x87','Luka Doncic')
player_positions['name'] = player_positions['name'].str.replace('Dereck Lively II','Dereck Lively')
player_positions['name'] = player_positions['name'].str.replace('P.J. Washington','PJ Washington')
player_positions['name'] = player_positions['name'].str.replace('Vlatko Ä\x8canÄ\x8dar','Vlatko Cancar')
player_positions['name'] = player_positions['name'].str.replace('Nikola JokiÄ\x87','Nikola Jokic')
player_positions['name'] = player_positions['name'].str.replace('Dario Å\xa0ariÄ\x87','Dario Saric')
player_positions['name'] = player_positions['name'].str.replace('Tim Hardaway','Tim Hardaway Jr')
player_positions['name'] = player_positions['name'].str.replace('Ron Holland','Ronald Holland')
player_positions['name'] = player_positions['name'].str.replace('Lindy Waters III','Lindy Waters')
player_positions['name'] = player_positions['name'].str.replace('Gary Payton II','Gary Payton')
player_positions['name'] = player_positions['name'].str.replace('T.J. McConnell','TJ McConnell')
player_positions['name'] = player_positions['name'].str.replace('Jeenathan Williams','Nate Williams')
player_positions['name'] = player_positions['name'].str.replace('Nikola JoviÄ\x87','Nikola Jovic')
player_positions['name'] = player_positions['name'].str.replace('A.J. Green','AJ Green')
player_positions['name'] = player_positions['name'].str.replace('Terrence Shannon ','Terrence Shannon')
player_positions['name'] = player_positions['name'].str.replace('Armel TraorÃ©','Armel Traore')
player_positions['name'] = player_positions['name'].str.replace('Karlo MatkoviÄ\x87','Karlo Matkovic')
player_positions['name'] = player_positions['name'].str.replace('CJ McCollum','C.J. McCollum')
player_positions['name'] = player_positions['name'].str.replace("Jae'Sean Tate","Jae'sean Tate")
player_positions['name'] = player_positions['name'].str.replace('Ricky Council IV','Ricky Council')
player_positions['name'] = player_positions['name'].str.replace('D.J. Steward','DJ Steward')
player_positions['name'] = player_positions['name'].str.replace('A.J. Lawson','AJ Lawson')
player_positions['name'] = player_positions['name'].str.replace('Marvin Bagley III','Marvin Bagley')
player_positions['name'] = player_positions['name'].str.replace('Jonas ValanÄ\x8diÅ«nas','Jonas Valanciunas')
player_positions['name'] = player_positions['name'].str.replace('Lester QuiÃ±ones','Lester Quinones')
player_positions['name'] = player_positions['name'].str.replace('Tristan Da Silva','Tristan da Silva')
player_positions['name'] = player_positions['name'].str.replace('Jusuf NurkiÄ\x87','Jusuf Nurkic')
player_positions['name'] = player_positions['name'].str.replace('Trey Murphy III','Trey Murphy')
player_positions['name'] = player_positions['name'].str.replace('Dennis SchrÃ¶der','Dennis Schroder')
player_positions['name'] = player_positions['name'].str.replace('Dennis SchrÃ¶der','Dennis Schroder')

In [31]:
player_positions[player_positions['name'].str.contains('Stew')].name.unique()

array(['Isaiah Stewart'], dtype=object)

In [32]:
u1 = players.merge(player_positions,how='left')

In [33]:
u1[u1['position'].isna()]

,code,name,team,team-code,position
18,89072992,Hugo Gonzalez,Boston,BOS,NaN
32,89050775,Nolan Traore,Brooklyn,BRK,NaN
33,1223837,Nicolas Claxton,Brooklyn,BRK,NaN
156,40722931,Alperen Sengun,Houston,HOU,NaN
207,89073019,Chris Manon,L.A. Lakers,LAL,NaN
218,52833056,GG Jackson,Memphis,MEM,NaN
405,58076075,David Jones Garcia,San Antonio,SAS,NaN
445,58092813,Clayton Carrington,Washington,WAS,NaN


In [34]:
players = players.merge(player_positions,how='left')

In [35]:
players = players[~players['position'].isna()]

In [36]:
players

,code,name,team,team-code,position
0,1218683,Nickeil Alexander-Walker,Atlanta,ATL,SG
1,47267142,Dyson Daniels,Atlanta,ATL,SG
2,58075832,N'Faly Dante,Atlanta,ATL,C
3,52566257,Mouhamed Gueye,Atlanta,ATL,PF
4,47267156,Caleb Houstan,Atlanta,ATL,SF
...,...,...,...,...,...
456,89072829,Will Riley,Washington,WAS,SF
457,58076218,Alex Sarr,Washington,WAS,C
458,55842866,Tristan Vukcevic,Washington,WAS,C
459,89072830,Jamir Watkins,Washington,WAS,SG


In [37]:
df = stats.merge(players,how='left')

In [38]:
df['minutes'] = df['minutes'].astype(float)

In [39]:
df['gameday'] = pd.to_datetime(df['gameday'])

In [40]:
df = df.sort_values('gameday',ascending=False)

In [41]:
df = df[df['minutes'] != 0]

In [42]:
df["rank"] = df.groupby("name")["gameday"].rank(method="dense", ascending=False)

In [43]:
df.name.nunique()

414

In [44]:
df.head()

,id,game-code,gameday,opponent,opponent-team-code,winorloss,result,statline,minutes,points,rebounds,assists,code,name,team,team-code,position,rank
3415,12468689,1437752,2025-11-13,Atlanta Hawks,ATL,L,"L, 122-132",25m 1r 3a,25.0,0,1,3,51625886,Svi Mykhailiuk,Utah,UTA,SF,1.0
56,12468676,1437752,2025-11-13,Utah Jazz,UTA,W,"W, 132-122",41m 31p 18r 14a,41.0,31,18,14,40722898,Jalen Johnson,Atlanta,ATL,SF,1.0
2765,12468743,1437751,2025-11-13,Indiana Pacers,IND,W,"W, 133-98",26m 17p 7r 3b,26.0,17,7,0,58076092,Oso Ighodaro,Phoenix,PHO,PF,1.0
710,12468718,1437750,2025-11-13,Toronto Raptors,TOR,L,"L, 113-126",2m,2.0,0,0,0,58075748,Luke Travers,Cleveland,CLE,SG,1.0
3289,12468703,1437750,2025-11-13,Cleveland Cavaliers,CLE,W,"W, 126-113",32m 25p 1r 6a,32.0,25,1,6,10021264,Immanuel Quickley,Toronto,TOR,PG,1.0


In [45]:
todays_game_links = get_game_links()

In [52]:
todays_game_links

['https://www.bovada.lv/sports/basketball/nba/phoenix-suns-golden-state-warriors-202511042200',
 'https://www.bovada.lv/sports/basketball/nba/oklahoma-city-thunder-l-a-clippers-202511042300',
 'https://www.bovada.lv/sports/basketball/nba/brooklyn-nets-orlando-magic-202511141900',
 'https://www.bovada.lv/sports/basketball/nba/miami-heat-new-york-knicks-202511141900',
 'https://www.bovada.lv/sports/basketball/nba/philadelphia-76ers-detroit-pistons-202511141930',
 'https://www.bovada.lv/sports/basketball/nba/charlotte-hornets-milwaukee-bucks-202511142000',
 'https://www.bovada.lv/sports/basketball/nba/los-angeles-lakers-new-orleans-pelicans-202511142000',
 'https://www.bovada.lv/sports/basketball/nba/portland-trail-blazers-houston-rockets-202511142000',
 'https://www.bovada.lv/sports/basketball/nba/sacramento-kings-minnesota-timberwolves-202511142000',
 'https://www.bovada.lv/sports/basketball/nba/l-a-clippers-dallas-mavericks-202511142030',
 'https://www.bovada.lv/sports/basketball/nba/g

In [46]:
props = pd.DataFrame()
for game_link in todays_game_links:
    props = pd.concat([props,get_player_props(game_link)])

In [47]:
old_props = pd.read_csv('Historical_Props.csv')

C:\Users\jtmck\AppData\Local\Temp\ipykernel_127996\2440907755.py:1: DtypeWarning: Columns (85) have mixed types. Specify dtype option on import or set low_memory=False.
  old_props = pd.read_csv('Historical_Props.csv')


In [48]:
props['date'] = str(today.year) + '-' + str(today.month) + '-' + str(today.day)

In [49]:
pd.concat([props,old_props]).to_csv('Historical_Props.csv')

In [51]:
props

,date


In [50]:
props.team.unique()

AttributeError: 'DataFrame' object has no attribute 'team'

In [ ]:
df.head()

In [ ]:
df.to_csv("Stats.csv")

In [ ]:
df

In [ ]:
props = props.rename(columns={'player':'name'})

In [ ]:
q1 = props.groupby(['name','type']).size().reset_index()

In [ ]:
props = props.groupby(['type','name','team'])['spread'].max().reset_index()

In [ ]:
props_pivot = props.pivot(index=['name'], columns='type', values='spread').reset_index()

In [ ]:
props_pivot

In [ ]:
df.head()

In [ ]:
df_pivot = df.merge(props_pivot,how='left')

In [ ]:
df_pivot.to_csv('Stats.csv')

In [ ]:
props = props.rename(columns={'player':'name'})

In [ ]:
props['spread'] = props['spread'].astype(float)

In [ ]:
df['points'] = df['points'].astype(float)
df['rebounds'] = df['rebounds'].astype(float)
df['assists'] = df['assists'].astype(float)

# Define Function

In [ ]:
def show_stat_df(stat,prop):
    stat_props = props[props['type'] == prop]
    spread_merge = df[df['minutes'] != 0].merge(stat_props[['name','spread']],how='left')
    spread_merge = spread_merge[~spread_merge['spread'].isna()]
    spread_merge = spread_merge.groupby('name').apply(lambda group: (group[stat] > group['spread']).mean() * 100).reset_index()
    spread_merge[0] = spread_merge[0].round(1)
    spread_merge = spread_merge.rename(columns={0:'hit%'})
    df['pra'] = df['points'] + df['rebounds'] + df['assists']
    d2 = df.groupby(['name','team-code','position'])[stat].mean().reset_index()
    df5 = df[df['rank'] <= 5]
    df10 = df[df['rank'] <= 10]
    df5 = df5.groupby(['name','team-code','position'])[stat].mean().reset_index()
    df10 = df10.groupby(['name','team-code','position'])[stat].mean().reset_index()
    df5 = df5.rename(columns={stat:stat+'_5g'})
    df10 = df10.rename(columns={stat:stat+'_10g'})
    p1 = df[df['minutes'] != 0]
    stat_props = stat_props.merge(d2,how='left').merge(df5,how='left')
    stat_props = stat_props.merge(df10,how='left').merge(spread_merge,how='left')
    stat_props = stat_props.merge(p1.groupby('name')[[stat]].min().reset_index().rename(columns={stat:stat+'_min'}),how='left')
    stat_props['delta'] = stat_props[stat] - stat_props['spread']
    
    stat_props['delta_5g'] = stat_props[stat+'_5g'] - stat_props['spread']
    stat_props['delta_10g'] = stat_props[stat+'_10g'] - stat_props['spread']
    
    stat_props = stat_props[~stat_props['delta'].isna()]
    stat_props['opponent'] = stat_props['team-code'].apply(lambda x: todays_games[x] if x in list(todays_games.keys()) else '')
    defense['team'] = defense['team'].apply(lambda x: 'GSW' if x == 'GS' else 'BRK' if x == 'BKN' else 'SAS' if x == 'SA' else 'NOP' if x == 'NO' else 'NYK' if x == 'NY' else 'CHH' if x == 'CHA' else x)
    defense_stat = defense[defense['stat'] == stat]
    defense_stat = defense_stat[['position','team','rank']]
    defense_stat = defense_stat.rename(columns={'team':'opponent'})
    stat_props = stat_props.merge(defense_stat,how='left')
    dev_stat = df.groupby(['name'])[stat].std().reset_index()
    dev_stat = dev_stat.rename(columns={stat:'std_dev'})
    stat_props = stat_props.merge(dev_stat,how='left')
    stat_props['std%'] = stat_props['std_dev'] / stat_props[stat]
    df['spm'] = df[stat] / df['minutes']
    spm_df = df.groupby(['name'])['spm'].mean().reset_index()
    stat_props = stat_props.merge(spm_df,how='left')
    pd.options.display.max_rows = 100

    matchup_df = stat_props[['name','team','opponent']]

    history = pd.read_csv('Historical_Data.csv')
    
    history = history.rename(columns={'player':'name','team_code':'team','opponent_code':'opponent','pts':'points','reb':'rebounds','ast':'assists','min':'minutes'})
    
    spread_merge = history[history['minutes'] != 0].merge(stat_props[['name','spread']],how='left')
    
    spread_merge = spread_merge[~spread_merge['spread'].isna()]
    
    opponent_percent = spread_merge.merge(matchup_df, on=['name', 'opponent'])
    spread_merge = spread_merge.rename(columns={'player':'name','team_code':'team','opponent_code':'opponent','pts':'points','reb':'rebounds','ast':'assists','min':'minutes'})
    spread_merge = spread_merge.groupby('name').apply(lambda group: (group[stat] > group['spread']).mean() * 100).reset_index()
    spread_merge[0] = spread_merge[0].round(1)
    spread_merge = spread_merge.rename(columns={0:'history_hit%'})
    stat_props = stat_props.merge(spread_merge,how='left')

    columns = list(stat_props.columns)

    # Find the index of the column to place it after
    after_index = columns.index('hit%')
    
    # Remove the column to move and reinsert it after the specified column
    columns.remove('history_hit%')
    columns.insert(after_index + 1, 'history_hit%')
    
    # Reorder the DataFrame
    stat_props = stat_props[columns]


    
    return stat_props

# Points

In [ ]:
points = show_stat_df('points','Total Points')

In [ ]:
points.sort_values('history_hit%',ascending=False)

In [ ]:
points[(points['delta'] > 0) & (points['delta_5g'] > 0) & (points['delta_10g'] > 0) & (points['hit%'] > 50) & (points['history_hit%'] > 50)]

In [ ]:
points[(points['delta'] > 0) & (points['delta_5g'] > 0) & (points['delta_10g'] > 0)]

In [ ]:
points[(points['delta'] < 0) & (points['delta_5g'] < 0) & (points['delta_10g'] < 0) & (points['hit%'] < 50) & (points['history_hit%'] < 50)]

# Rebounds

In [82]:
rebounds = show_stat_df('rebounds','Total Rebounds')

C:\Users\jtmck\AppData\Local\Temp\ipykernel_36272\1563515617.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  spread_merge = spread_merge.groupby('name').apply(lambda group: (group[stat] > group['spread']).mean() * 100).reset_index()
C:\Users\jtmck\AppData\Local\Temp\ipykernel_36272\1563515617.py:43: DtypeWarning: Columns (4,15,25,26,28,29,62,63,64) have mixed types. Specify dtype option on import or set low_memory=False.
  history = pd.read_csv('Historical_Data.csv')
C:\Users\jtmck\AppData\Local\Temp\ipykernel_36272\1563515617.py:53: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns 

In [83]:
rebounds

,type,name,team,spread,team-code,position,rebounds,rebounds_5g,rebounds_10g,hit%,history_hit%,rebounds_min,delta,delta_5g,delta_10g,opponent,rank,std_dev,std%,spm
0,Total Rebounds,Anthony Edwards,MIN,6.5,MIN,SG,5.966292,7.6,8.0,33.7,29.3,0.0,-0.533708,1.1,1.5,OKC,29.0,2.634774,0.441610,0.162288
1,Total Rebounds,Chet Holmgren,OKC,8.5,OKC,C,8.465116,11.8,9.7,48.8,42.6,1.0,-0.034884,3.3,1.2,MIN,7.0,4.055266,0.479056,0.306766
2,Total Rebounds,Isaiah Hartenstein,OKC,8.5,OKC,C,10.376812,8.8,8.8,69.6,28.3,2.0,1.876812,0.3,0.3,MIN,7.0,3.540112,0.341156,0.375864
3,Total Rebounds,Jaden McDaniels,MIN,5.5,MIN,PF,5.793478,5.8,6.1,51.1,26.0,0.0,0.293478,0.3,0.6,OKC,8.0,3.036547,0.524132,0.175737
4,Total Rebounds,Jalen Williams,OKC,4.5,OKC,SG,5.345679,5.4,5.4,64.2,54.5,1.0,0.845679,0.9,0.9,MIN,6.0,2.007240,0.375488,0.167060
5,Total Rebounds,Josh Hart,NYK,9.5,NYK,SG,9.438202,7.8,8.8,49.4,26.6,2.0,-0.061798,-1.7,-0.7,,NaN,3.592780,0.380664,0.250676
6,Total Rebounds,Julius Randle,MIN,7.5,MIN,PF,6.911392,6.6,5.9,36.7,65.5,1.0,-0.588608,-0.9,-1.6,OKC,8.0,2.450483,0.354557,0.209953
7,Total Rebounds,Karl-Anthony Towns,NYK,11.5,NYK,C,12.571429,12.6,11.9,53.6,42.1,6.0,1.071429,1.1,0.4,,NaN,3.850476,0.306288,0.360318
8,Total Rebounds,Mitchell Robinson,NYK,7.5,NYK,C,6.310345,8.2,6.9,31.0,48.6,2.0,-1.189655,0.7,-0.6,,NaN,3.230474,0.511933,0.348039
9,Total Rebounds,OG Anunoby,NYK,5.5,NYK,PF,4.860465,5.4,4.8,33.7,28.6,0.0,-0.639535,-0.1,-0.7,,NaN,2.260467,0.465072,0.130427


In [84]:
rebounds[(rebounds['delta'] > 0) & (rebounds['delta_5g'] > 0) & (rebounds['delta_10g'] > 0) & (rebounds['hit%'] > 50) & (rebounds['history_hit%'] > 50)]

,type,name,team,spread,team-code,position,rebounds,rebounds_5g,rebounds_10g,hit%,history_hit%,rebounds_min,delta,delta_5g,delta_10g,opponent,rank,std_dev,std%,spm
4,Total Rebounds,Jalen Williams,OKC,4.5,OKC,SG,5.345679,5.4,5.4,64.2,54.5,1.0,0.845679,0.9,0.9,MIN,6.0,2.00724,0.375488,0.16706


In [85]:
rebounds[(rebounds['delta'] > 0) & (rebounds['delta_5g'] > 0) & (rebounds['delta_10g'] > 0)]

,type,name,team,spread,team-code,position,rebounds,rebounds_5g,rebounds_10g,hit%,history_hit%,rebounds_min,delta,delta_5g,delta_10g,opponent,rank,std_dev,std%,spm
2,Total Rebounds,Isaiah Hartenstein,OKC,8.5,OKC,C,10.376812,8.8,8.8,69.6,28.3,2.0,1.876812,0.3,0.3,MIN,7.0,3.540112,0.341156,0.375864
3,Total Rebounds,Jaden McDaniels,MIN,5.5,MIN,PF,5.793478,5.8,6.1,51.1,26.0,0.0,0.293478,0.3,0.6,OKC,8.0,3.036547,0.524132,0.175737
4,Total Rebounds,Jalen Williams,OKC,4.5,OKC,SG,5.345679,5.4,5.4,64.2,54.5,1.0,0.845679,0.9,0.9,MIN,6.0,2.007240,0.375488,0.167060
7,Total Rebounds,Karl-Anthony Towns,NYK,11.5,NYK,C,12.571429,12.6,11.9,53.6,42.1,6.0,1.071429,1.1,0.4,,NaN,3.850476,0.306288,0.360318


In [86]:
rebounds[(rebounds['delta'] < 0) & (rebounds['delta_5g'] < 0) & (rebounds['delta_10g'] < 0) & (rebounds['hit%'] < 50) & (rebounds['history_hit%'] < 50)]

,type,name,team,spread,team-code,position,rebounds,rebounds_5g,rebounds_10g,hit%,history_hit%,rebounds_min,delta,delta_5g,delta_10g,opponent,rank,std_dev,std%,spm
5,Total Rebounds,Josh Hart,NYK,9.5,NYK,SG,9.438202,7.8,8.8,49.4,26.6,2.0,-0.061798,-1.7,-0.7,,NaN,3.592780,0.380664,0.250676
9,Total Rebounds,OG Anunoby,NYK,5.5,NYK,PF,4.860465,5.4,4.8,33.7,28.6,0.0,-0.639535,-0.1,-0.7,,NaN,2.260467,0.465072,0.130427


# Assists

In [87]:
assists = show_stat_df('assists','Total Assists')

C:\Users\jtmck\AppData\Local\Temp\ipykernel_36272\1563515617.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  spread_merge = spread_merge.groupby('name').apply(lambda group: (group[stat] > group['spread']).mean() * 100).reset_index()
C:\Users\jtmck\AppData\Local\Temp\ipykernel_36272\1563515617.py:43: DtypeWarning: Columns (4,15,25,26,28,29,62,63,64) have mixed types. Specify dtype option on import or set low_memory=False.
  history = pd.read_csv('Historical_Data.csv')
C:\Users\jtmck\AppData\Local\Temp\ipykernel_36272\1563515617.py:53: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns 

In [88]:
assists[(assists['delta'] > 0) & (assists['delta_5g'] > 0) & (assists['delta_10g'] > 0) & (assists['hit%'] > 50) & (assists['history_hit%'] > 50)]

,type,name,team,spread,team-code,position,assists,assists_5g,assists_10g,hit%,history_hit%,assists_min,delta,delta_5g,delta_10g,opponent,rank,std_dev,std%,spm
2,Total Assists,Donte DiVincenzo,MIN,2.5,MIN,SG,3.611111,3.6,3.5,66.7,51.4,0.0,1.111111,1.1,1.0,OKC,9.0,2.010925,0.556871,0.140312


In [89]:
assists[(assists['delta'] > 0) & (assists['delta_5g'] > 0) & (assists['delta_10g'] > 0)]

,type,name,team,spread,team-code,position,assists,assists_5g,assists_10g,hit%,history_hit%,assists_min,delta,delta_5g,delta_10g,opponent,rank,std_dev,std%,spm
0,Total Assists,Alex Caruso,OKC,2.5,OKC,SG,2.630769,2.6,3.0,44.6,50.1,0.0,0.130769,0.1,0.5,MIN,9.0,1.644776,0.625207,0.131602
2,Total Assists,Donte DiVincenzo,MIN,2.5,MIN,SG,3.611111,3.6,3.5,66.7,51.4,0.0,1.111111,1.1,1.0,OKC,9.0,2.010925,0.556871,0.140312
5,Total Assists,Jalen Williams,OKC,4.5,OKC,SG,5.197531,6.2,5.7,60.5,43.7,1.0,0.697531,1.7,1.2,MIN,9.0,1.964814,0.378028,0.162804


In [90]:
assists[(assists['delta'] < 0) & (assists['delta_5g'] < 0) & (assists['delta_10g'] < 0) & (assists['hit%'] < 50) & (assists['history_hit%'] < 50)]

,type,name,team,spread,team-code,position,assists,assists_5g,assists_10g,hit%,history_hit%,assists_min,delta,delta_5g,delta_10g,opponent,rank,std_dev,std%,spm


In [91]:
assists.sort_values('history_hit%',ascending=False)

,type,name,team,spread,team-code,position,assists,assists_5g,assists_10g,hit%,history_hit%,assists_min,delta,delta_5g,delta_10g,opponent,rank,std_dev,std%,spm
11,Total Assists,Tyrese Haliburton,IND,8.5,IND,PG,9.228916,7.0,9.3,56.6,52.1,2.0,0.728916,-1.5,0.8,,NaN,3.228400,0.349814,0.274549
2,Total Assists,Donte DiVincenzo,MIN,2.5,MIN,SG,3.611111,3.6,3.5,66.7,51.4,0.0,1.111111,1.1,1.0,OKC,9.0,2.010925,0.556871,0.140312
0,Total Assists,Alex Caruso,OKC,2.5,OKC,SG,2.630769,2.6,3.0,44.6,50.1,0.0,0.130769,0.1,0.5,MIN,9.0,1.644776,0.625207,0.131602
9,Total Assists,Pascal Siakam,IND,3.5,IND,PF,3.340909,3.8,3.1,44.3,45.6,0.0,-0.159091,0.3,-0.4,,NaN,1.773926,0.530971,0.103090
5,Total Assists,Jalen Williams,OKC,4.5,OKC,SG,5.197531,6.2,5.7,60.5,43.7,1.0,0.697531,1.7,1.2,MIN,9.0,1.964814,0.378028,0.162804
3,Total Assists,Isaiah Hartenstein,OKC,2.5,OKC,C,3.608696,2.0,2.6,69.6,31.5,0.0,1.108696,-0.5,0.1,MIN,13.0,1.832849,0.507898,0.133061
8,Total Assists,Mikal Bridges,NYK,3.5,NYK,SF,3.645161,3.2,3.2,47.3,29.2,0.0,0.145161,-0.3,-0.3,,NaN,2.046445,0.561414,0.098468
10,Total Assists,Shai Gilgeous-Alexander,OKC,6.5,OKC,PG,6.329545,6.0,6.5,47.7,28.9,1.0,-0.170455,-0.5,0.0,MIN,9.0,2.211369,0.349372,0.186582
1,Total Assists,Anthony Edwards,MIN,5.5,MIN,SG,4.696629,5.6,5.9,33.7,27.8,0.0,-0.803371,0.1,0.4,OKC,9.0,2.332598,0.496654,0.128972
7,Total Assists,Julius Randle,MIN,5.5,MIN,PF,4.835443,7.4,5.9,34.2,21.9,1.0,-0.664557,1.9,0.4,OKC,5.0,2.311789,0.478093,0.146451


In [92]:
points_props.sort_values('delta_5g',ascending=False).head(100)

NameError: name 'points_props' is not defined

# Points

In [ ]:
points_props = props[props['type'] == 'Total Points']

In [63]:
d1 = df.copy()

In [64]:
spread_merge = df[df['minutes'] != 0].merge(points_props[['name','spread']],how='left')

In [65]:
spread_merge = spread_merge[~spread_merge['spread'].isna()]

In [66]:
spread_merge = spread_merge.groupby('name').apply(lambda group: (group['points'] > group['spread']).mean() * 100).reset_index()

C:\Users\jtmck\AppData\Local\Temp\ipykernel_10048\2933933359.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  spread_merge = spread_merge.groupby('name').apply(lambda group: (group['points'] > group['spread']).mean() * 100).reset_index()


In [67]:
spread_merge[0] = spread_merge[0].round(1)

In [68]:
spread_merge = spread_merge.rename(columns={0:'hit%'})

In [69]:
d1['pra'] = d1['points'] + d1['rebounds'] + d1['assists']
df['pra'] = df['points'] + df['rebounds'] + df['assists']

In [70]:
d2 = d1.groupby(['name','team-code','position'])['points'].mean().reset_index()

In [71]:
df5 = d1[d1['rank'] <= 5]

In [72]:
df10 = d1[d1['rank'] <= 10]

In [73]:
df5 = df5.groupby(['name','team-code','position'])['points'].mean().reset_index()
df10 = df10.groupby(['name','team-code','position'])['points'].mean().reset_index()

In [74]:
df5 = df5.rename(columns={'points':'points_5g'})
df10 = df10.rename(columns={'points':'points_10g'})

In [75]:
p1 = df[df['minutes'] != 0]

In [76]:
points_props = points_props.merge(d2,how='left').merge(df5,how='left')
points_props = points_props.merge(df10,how='left').merge(spread_merge,how='left')
points_props = points_props.merge(p1.groupby('name')[['points']].min().reset_index().rename(columns={'points':'points_min'}),how='left')

In [77]:
points_props.head()

,type,name,team,spread,date,team-code,position,points,points_5g,points_10g,hit%,points_min
0,Total Points,Ausar Thompson,DET,8.5,2025-1-16,DET,SF,7.611111,8.0,7.5,38.9,3.0
1,Total Points,Cade Cunningham,DET,25.5,2025-1-16,DET,PG,24.527778,27.0,26.2,41.7,13.0
2,Total Points,Jalen Duren,DET,9.5,2025-1-16,DET,C,9.513514,10.0,10.8,54.1,0.0
3,Total Points,Malik Beasley,DET,16.5,2025-1-16,DET,SG,16.475000,17.8,16.2,57.5,2.0
4,Total Points,Myles Turner,IND,14.5,2025-1-16,IND,C,15.052632,14.6,14.3,47.4,7.0


In [78]:
points_props['delta'] = points_props['points'] - points_props['spread']

In [79]:
points_props['delta_5g'] = points_props['points_5g'] - points_props['spread']
points_props['delta_10g'] = points_props['points_10g'] - points_props['spread']

In [80]:
points_props = points_props[~points_props['delta'].isna()]

In [81]:
points_props['team-code'].unique()

array(['DET', 'IND', 'OKC', 'CLE', 'HOU'], dtype=object)

In [82]:
points_props[points_props['team-code'] == 'MIL']

,type,name,team,spread,date,team-code,position,points,points_5g,points_10g,hit%,points_min,delta,delta_5g,delta_10g


In [83]:
points_props['team-code'].unique()

array(['DET', 'IND', 'OKC', 'CLE', 'HOU'], dtype=object)

In [84]:
points_props['opponent'] = points_props['team-code'].apply(lambda x: todays_games[x] if x in list(todays_games.keys()) else '')

In [85]:
defense['team'] = defense['team'].apply(lambda x: 'GSW' if x == 'GS' else 'BRK' if x == 'BKN' else 'SAS' if x == 'SA' else 'NOP' if x == 'NO' else 'NYK' if x == 'NY' else 'CHH' if x == 'CHA' else x)

In [86]:
defense_points = defense[defense['stat'] == 'points']

In [87]:
defense_points = defense_points[['position','team','rank']]

In [88]:
defense_points = defense_points.rename(columns={'team':'opponent'})

In [89]:
points_props = points_props.merge(defense_points,how='left')

In [90]:
dev_points = df.groupby(['name'])['points'].std().reset_index()

In [91]:
dev_points = dev_points.rename(columns={'points':'std_dev'})

In [92]:
points_props = points_props.merge(dev_points,how='left')

In [93]:
points_props['std%'] = points_props['std_dev'] / points_props['points']

In [94]:
df['ppm'] = df['points'] / df['minutes']

In [95]:
ppm_df = df.groupby(['name'])['ppm'].mean().reset_index()

In [96]:
points_props = points_props.merge(ppm_df,how='left')

In [97]:
pd.options.display.max_rows = 100

In [98]:
points_props.shape

(25, 20)

In [99]:
defense.team.unique()

array(['DET', 'NOP', 'CLE', 'SAS', 'LAL', 'MIL', 'ORL', 'LAC', 'SAC',
       'ATL', 'PHI', 'MIN', 'DAL', 'TOR', 'BRK', 'OKC', 'NYK', 'BOS',
       'CHH', 'HOU', 'PHO', 'DEN', 'IND', 'MEM', 'CHI', 'POR', 'WAS',
       'GSW', 'MIA', 'UTA'], dtype=object)

In [100]:
points_props.sort_values('delta_5g',ascending=False).head(100)

,type,name,team,spread,date,team-code,position,points,points_5g,points_10g,hit%,points_min,delta,delta_5g,delta_10g,opponent,rank,std_dev,std%,ppm
24,Total Points,Jalen Green,HOU,22.5,2025-1-16,HOU,SG,21.333333,33.0,27.7,38.5,6.0,-1.166667,10.5,5.2,SAC,24,9.592578,0.449652,0.640455
11,Total Points,Darius Garland,CLE,19.5,2025-1-16,CLE,PG,21.026316,25.4,23.0,57.9,7.0,1.526316,5.9,3.5,OKC,2,7.681099,0.365309,0.689261
16,Total Points,Jarrett Allen,CLE,13.5,2025-1-16,CLE,C,14.128205,17.8,16.8,51.3,2.0,0.628205,4.3,3.3,OKC,11,6.262585,0.443268,0.483529
8,Total Points,Aaron Wiggins,OKC,8.5,2025-1-16,OKC,SG,9.375000,12.6,13.1,52.5,0.0,0.875000,4.1,4.6,CLE,10,5.102476,0.544264,0.447653
14,Total Points,Isaiah Joe,OKC,8.5,2025-1-16,OKC,SG,8.736842,12.4,10.1,44.7,0.0,0.236842,3.9,1.6,CLE,10,5.754830,0.658685,0.410117
10,Total Points,Cason Wallace,OKC,9.5,2025-1-16,OKC,SG,7.333333,13.2,9.2,25.6,0.0,-2.166667,3.7,-0.3,CLE,10,4.590226,0.625940,0.263930
21,Total Points,Cam Whitmore,HOU,10.5,2025-1-16,HOU,SF,10.095238,13.2,13.3,42.9,4.0,-0.404762,2.7,2.8,SAC,15,6.355350,0.629539,0.677550
18,Total Points,Max Strus,CLE,8.5,2025-1-16,CLE,SF,8.181818,10.8,8.1,54.5,2.0,-0.318182,2.3,-0.4,OKC,12,5.510321,0.673484,0.346024
23,Total Points,Fred VanVleet,HOU,14.5,2025-1-16,HOU,PG,15.000000,16.2,15.3,51.4,2.0,0.500000,1.7,0.8,SAC,8,7.753136,0.516876,0.434424
1,Total Points,Cade Cunningham,DET,25.5,2025-1-16,DET,PG,24.527778,27.0,26.2,41.7,13.0,-0.972222,1.5,0.7,IND,22,6.500488,0.265026,0.691870


In [101]:
points_props[points_props['team'] == 'HOU']

,type,name,team,spread,date,team-code,position,points,points_5g,points_10g,hit%,points_min,delta,delta_5g,delta_10g,opponent,rank,std_dev,std%,ppm
20,Total Points,Amen Thompson,HOU,15.5,2025-1-16,HOU,SF,12.567568,16.6,14.6,32.4,1.0,-2.932432,1.1,-0.9,SAC,15,5.649683,0.449545,0.422711
21,Total Points,Cam Whitmore,HOU,10.5,2025-1-16,HOU,SF,10.095238,13.2,13.3,42.9,4.0,-0.404762,2.7,2.8,SAC,15,6.355350,0.629539,0.677550
22,Total Points,Dillon Brooks,HOU,11.5,2025-1-16,HOU,SF,13.027778,7.2,14.0,55.6,0.0,1.527778,-4.3,2.5,SAC,15,7.004024,0.537622,0.411383
23,Total Points,Fred VanVleet,HOU,14.5,2025-1-16,HOU,PG,15.000000,16.2,15.3,51.4,2.0,0.500000,1.7,0.8,SAC,8,7.753136,0.516876,0.434424
24,Total Points,Jalen Green,HOU,22.5,2025-1-16,HOU,SG,21.333333,33.0,27.7,38.5,6.0,-1.166667,10.5,5.2,SAC,24,9.592578,0.449652,0.640455


In [102]:
# Points Dev

In [103]:
all_games = df[['opponent-team-code','points','name','team-code','position']].copy()

In [104]:
all_games.name.nunique()

506

In [105]:
all_games = all_games.rename(columns={'team-code':'team','opponent-team-code':'opponent'})

In [106]:
avg = all_games.groupby(['name','team'])['points'].mean().reset_index()

In [107]:
avg = avg.rename(columns={'points':'avg_pts'})

In [108]:
all_games = all_games.merge(avg,how='left')

In [109]:
all_games['delta'] = all_games['points'] - all_games['avg_pts']

In [110]:
all_games

,opponent,points,name,team,position,avg_pts,delta
0,MIN,7.0,Gary Payton,GSW,SG,4.724138,2.275862
1,LAL,7.0,Duncan Robinson,MIA,SF,10.513514,-3.513514
2,TOR,18.0,Kristaps Porzingis,BOS,C,18.529412,-0.529412
3,PHI,17.0,OG Anunoby,NYK,PF,16.000000,1.000000
4,HOU,10.0,Julian Strawther,DEN,SG,9.275000,0.725000
...,...,...,...,...,...,...,...
13228,BOS,3.0,Pacome Dadiet,NYK,SG,1.583333,1.416667
13229,MIN,9.0,D'Angelo Russell,BRK,PG,12.379310,-3.379310
13230,BOS,12.0,Karl-Anthony Towns,NYK,C,25.394737,-13.394737
13231,LAL,10.0,Donte DiVincenzo,MIN,SG,11.000000,-1.000000


In [111]:
all_games[(all_games['opponent'] == 'TOR') & (all_games['position'] == 'C')]['delta'].mean()

np.float64(0.5584020488437026)

In [112]:
median_defense = all_games.groupby(['opponent','position'])['delta'].median().reset_index().sort_values('delta',ascending=False)

In [113]:
median_defense

,opponent,position,delta
127,SAC,PG,2.800000
3,ATL,SF,1.868182
145,WAS,C,0.975000
115,PHO,C,0.823529
124,POR,SG,0.702703
...,...,...,...
125,SAC,C,-1.776709
27,CLE,PG,-1.815789
61,LAC,PF,-2.187500
131,SAS,PF,-2.250000


In [114]:
median_defense[median_defense['opponent'] == 'TOR']

,opponent,position,delta
137,TOR,PG,0.440000
139,TOR,SG,0.328125
135,TOR,C,-0.128205
136,TOR,PF,-0.291667
138,TOR,SF,-0.397436


# Points - Rebounds - Assists

In [115]:
props['type'].unique()

array(['Total Points', 'Total Rebounds', 'Total Made 3 Points Shots',
       'Total Assists', 'Total Points, Rebounds and Assists'],
      dtype=object)

In [116]:
pra = props[props['type'] == 'Total Points, Rebounds and Assists']

In [117]:
d1.head()

,id,game-code,gameday,opponent,opponent-team-code,winorloss,result,statline,minutes,points,rebounds,assists,code,name,team,team-code,position,rank,pra
4710,12155546,1081693,2025-01-15,Minnesota Timberwolves,MIN,W,"W, 116-115",19m 7p 6r 1a,19.0,7.0,6.0,1.0,124084,Gary Payton,Golden State,GSW,SG,1.0,14.0
7301,12155569,1081698,2025-01-15,Los Angeles Lakers,LAL,L,"L, 108-117",22m 7p 2r 1a 1b,22.0,7.0,2.0,1.0,664841,Duncan Robinson,Miami,MIA,SF,1.0,10.0
590,12155359,1081690,2025-01-15,Toronto Raptors,TOR,L,"L, 97-110",35m 18p 8r 3a 2b,35.0,18.0,8.0,3.0,3103,Kristaps Porzingis,Boston,BOS,C,1.0,29.0
8691,12155334,1081689,2025-01-15,Philadelphia 76ers,PHI,W,"W, 125-119",45m 17p 4r 4a,45.0,17.0,4.0,4.0,329870,OG Anunoby,New York,NYK,PF,1.0,25.0
3872,12155512,1081696,2025-01-15,Houston Rockets,HOU,L,"L, 108-128",25m 10p 1r 3a,25.0,10.0,1.0,3.0,52566239,Julian Strawther,Denver,DEN,SG,1.0,14.0


In [118]:
d2 = d1.groupby(['name','position'])['pra'].mean().reset_index()

In [119]:
df5 = df[df['rank'] <= 5]
df10 = df[df['rank'] <= 10]

In [120]:
df5 = df5.groupby('name')['pra'].mean().reset_index()
df10 = df10.groupby('name')['pra'].mean().reset_index()

In [121]:
df5 = df5.rename(columns={'pra':'pra_5g'})
df10 = df10.rename(columns={'pra':'pra_10g'})

In [122]:
pra = pra.merge(d2,how='left').merge(df5,how='left').merge(df10,how='left')

In [123]:
pra['spread'] = pra['spread'].astype(float)

In [124]:
pra['delta'] = pra['pra'] - pra['spread']
pra['delta_5g'] = pra['pra_5g'] - pra['spread']
pra['delta_10g'] = pra['pra_10g'] - pra['spread']

In [125]:
pra = pra[~pra['delta'].isna()]

In [126]:
pra = pra.rename(columns={'team':'team-code'})

In [127]:
pra['opponent'] = pra['team-code'].apply(lambda x: todays_games['PHO'] if x == 'PHX' else todays_games['GSW'] if x == 'GS' else todays_games['BRK'] if x == 'BKN' else todays_games['SAS'] if x == 'SA' else todays_games['NOP'] if x == 'NO' else todays_games['NYK'] if x == 'NY' else todays_games['CHH'] if x == 'CHA' else todays_games[x])

In [128]:
defense.stat.unique()

array(['blocks', '3pm', 'steals', 'rebounds', 'assists', 'points', 'fg%',
       'ft%'], dtype=object)

In [129]:
pra_defense = defense[(defense['stat'] == 'points') | (defense['stat'] == 'rebounds') | (defense['stat'] == 'assists')]

In [130]:
pra_defense = pra_defense.pivot(index=['position','team'], columns='stat', values='value').reset_index()

In [131]:
pra_defense['pra'] = pra_defense['points'] + pra_defense['rebounds'] + pra_defense['assists']

In [132]:
pra_defense = pra_defense.sort_values('pra')

In [133]:
pra_defense['rank'] = pra_defense.groupby(['position'])['pra'].rank(ascending=True, method='min')

In [134]:
pra_defense = pra_defense.sort_values('pra')

In [135]:
pra_defense = pra_defense.rename(columns={'team':'opponent'})

In [136]:
pra_defense.head()

stat,position,opponent,assists,points,rebounds,pra,rank
111,SF,ORL,3.9,17.8,7.3,17.87.33.9,1.0
16,C,MIL,3.9,18.0,14.5,18.014.53.9,1.0
19,C,NYK,4.3,18.9,13.7,18.913.74.3,2.0
15,C,MIA,4.2,18.9,16.3,18.916.34.2,3.0
100,SF,HOU,3.4,19.1,7.5,19.17.53.4,2.0


In [137]:
pra = pra.sort_values('delta',ascending=False).merge(pra_defense[['position','opponent','rank']],how='left')

In [138]:
pra.shape

(24, 14)

In [139]:
pra

,type,name,team-code,spread,date,position,pra,pra_5g,pra_10g,delta,delta_5g,delta_10g,opponent,rank
0,"Total Points, Rebounds and Assists",Caris LeVert,CLE,14.5,2025-1-16,SF,17.818182,15.6,15.2,3.318182,1.1,0.7,OKC,12.0
1,"Total Points, Rebounds and Assists",Donovan Mitchell,CLE,30.5,2025-1-16,SG,32.162162,27.6,30.6,1.662162,-2.9,0.1,OKC,1.0
2,"Total Points, Rebounds and Assists",Dillon Brooks,HOU,17.5,2025-1-16,SF,18.444444,12.8,19.8,0.944444,-4.7,2.3,SAC,16.0
3,"Total Points, Rebounds and Assists",Darius Garland,CLE,29.5,2025-1-16,PG,30.236842,33.4,33.0,0.736842,3.9,3.5,OKC,2.0
4,"Total Points, Rebounds and Assists",Aaron Wiggins,OKC,13.5,2025-1-16,SG,14.150000,18.8,18.8,0.650000,5.3,5.3,CLE,11.0
5,"Total Points, Rebounds and Assists",Fred VanVleet,HOU,24.5,2025-1-16,PG,25.135135,24.6,24.7,0.635135,0.1,0.2,SAC,8.0
6,"Total Points, Rebounds and Assists",Luguentz Dort,OKC,15.5,2025-1-16,SF,15.641026,14.8,15.0,0.141026,-0.7,-0.5,CLE,11.0
7,"Total Points, Rebounds and Assists",Myles Turner,IND,23.5,2025-1-16,C,23.631579,22.2,22.7,0.131579,-1.3,-0.8,DET,22.0
8,"Total Points, Rebounds and Assists",Malik Beasley,DET,21.5,2025-1-16,SG,21.375000,22.8,20.8,-0.125000,1.3,-0.7,IND,20.0
9,"Total Points, Rebounds and Assists",Jarrett Allen,CLE,26.5,2025-1-16,C,26.307692,32.2,29.6,-0.192308,5.7,3.1,OKC,12.0


# Rebounds

In [140]:
rebounds = props[props['type'] == 'Total Rebounds']

In [141]:
spread_merge = df[df['minutes'] != 0].merge(rebounds[['name','spread']],how='left')

In [142]:
spread_merge = spread_merge[~spread_merge['spread'].isna()]

In [143]:
spread_merge = spread_merge.groupby('name').apply(lambda group: (group['rebounds'] > group['spread']).mean() * 100).reset_index()

C:\Users\jtmck\AppData\Local\Temp\ipykernel_10048\3487771906.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  spread_merge = spread_merge.groupby('name').apply(lambda group: (group['rebounds'] > group['spread']).mean() * 100).reset_index()


In [144]:
spread_merge[0] = spread_merge[0].round(1)

In [145]:
spread_merge = spread_merge.rename(columns={0:'hit%'})

In [146]:
d1 = df.groupby(['name','position','team-code'])['rebounds'].mean().reset_index()

In [147]:
df5 = df[df['rank'] <= 5]
df10 = df[df['rank'] <= 10]

In [148]:
df5 = df5.groupby(['name'])['rebounds'].mean().reset_index()
df10 = df10.groupby(['name'])['rebounds'].mean().reset_index()

In [149]:
df5 = df5.rename(columns={'rebounds':'rebounds_5g'})
df10 = df10.rename(columns={'rebounds':'rebounds_10g'})

In [150]:
rebounds = rebounds.merge(d1,how='left').merge(df5,how='left').merge(df10,how='left').merge(spread_merge,how='left')

In [151]:
rebounds['delta'] = rebounds['rebounds'] - rebounds['spread']
rebounds['delta_5g'] = rebounds['rebounds_5g'] - rebounds['spread']
rebounds['delta_10g'] = rebounds['rebounds_10g'] - rebounds['spread']

In [152]:
rebounds = rebounds[~rebounds['rebounds'].isna()]

In [153]:
defense_rebounds = defense[defense['stat'] == 'rebounds']

In [154]:
defense_rebounds = defense_rebounds[['position','team','rank']]

In [155]:
defense_rebounds = defense_rebounds.rename(columns={'team':'opponent'})

In [156]:
rebounds.head()

,type,name,team,spread,date,position,team-code,rebounds,rebounds_5g,rebounds_10g,hit%,delta,delta_5g,delta_10g
0,Total Rebounds,Ausar Thompson,DET,5.5,2025-1-16,SF,DET,4.666667,5.6,5.5,27.8,-0.833333,0.1,0.0
1,Total Rebounds,Cade Cunningham,DET,5.5,2025-1-16,PG,DET,6.555556,5.8,5.4,66.7,1.055556,0.3,-0.1
2,Total Rebounds,Jalen Duren,DET,9.5,2025-1-16,C,DET,9.378378,10.0,9.9,43.2,-0.121622,0.5,0.4
3,Total Rebounds,Myles Turner,IND,6.5,2025-1-16,C,IND,7.052632,6.6,7.0,52.6,0.552632,0.1,0.5
4,Total Rebounds,Pascal Siakam,IND,7.5,2025-1-16,PF,IND,7.390244,7.8,8.3,43.9,-0.109756,0.3,0.8


In [157]:
defense_rebounds

,position,opponent,rank
188,PF,LAC,1
476,PF,MIL,1
676,PF,DEN,3
764,PF,UTA,4
532,PF,NOP,5
...,...,...,...
196,SF,MIL,28
388,SF,CHI,29
1020,SF,DET,29
940,PF,SAC,29


In [158]:
rebounds['opponent'] = rebounds['team'].apply(lambda x: todays_games[x] if x in list(todays_games.keys()) else '')

In [159]:
rebounds = rebounds.merge(defense_rebounds,how='left')

In [160]:
rebounds.sort_values('delta_10g',ascending=False)

,type,name,team,spread,date,position,team-code,rebounds,rebounds_5g,rebounds_10g,hit%,delta,delta_5g,delta_10g,opponent,rank
4,Total Rebounds,Pascal Siakam,IND,7.5,2025-1-16,PF,IND,7.390244,7.8,8.3,43.9,-0.109756,0.3,0.8,DET,5
10,Total Rebounds,Amen Thompson,HOU,9.5,2025-1-16,SF,HOU,7.756757,12.0,10.2,29.7,-1.743243,2.5,0.7,SAC,18
3,Total Rebounds,Myles Turner,IND,6.5,2025-1-16,C,IND,7.052632,6.6,7.0,52.6,0.552632,0.1,0.5,DET,2
2,Total Rebounds,Jalen Duren,DET,9.5,2025-1-16,C,DET,9.378378,10.0,9.9,43.2,-0.121622,0.5,0.4,IND,22
0,Total Rebounds,Ausar Thompson,DET,5.5,2025-1-16,SF,DET,4.666667,5.6,5.5,27.8,-0.833333,0.1,0.0,IND,11
1,Total Rebounds,Cade Cunningham,DET,5.5,2025-1-16,PG,DET,6.555556,5.8,5.4,66.7,1.055556,0.3,-0.1,IND,3
8,Total Rebounds,Jarrett Allen,CLE,10.5,2025-1-16,C,CLE,10.205128,11.2,10.4,48.7,-0.294872,0.7,-0.1,OKC,23
7,Total Rebounds,Jalen Williams,OKC,5.5,2025-1-16,SG,OKC,5.650000,5.0,5.2,50.0,0.150000,-0.5,-0.3,CLE,15
5,Total Rebounds,Tobias Harris,DET,6.5,2025-1-16,PF,DET,6.447368,5.4,6.1,55.3,-0.052632,-1.1,-0.4,IND,28
9,Total Rebounds,Shai Gilgeous-Alexander,OKC,5.5,2025-1-16,PG,OKC,5.475000,5.8,4.9,45.0,-0.025000,0.3,-0.6,CLE,3


In [161]:
dev_rebounds = df.groupby(['name'])['rebounds'].std().reset_index()

In [162]:
dev_rebounds = dev_rebounds.rename(columns={'rebounds':'std_dev'})

In [163]:
rebounds = rebounds.merge(dev_rebounds,how='left')

In [164]:
rebounds['std%'] = rebounds['rebounds']  / rebounds['std_dev']

In [165]:
df['rpm'] = df['rebounds'] / df['minutes']

In [166]:
rpm_df = df.groupby(['name'])['rpm'].mean().reset_index()

In [167]:
rebounds = rebounds.merge(rpm_df,how='left')

In [168]:
rebounds.sort_values('delta_10g',ascending=False)

,type,name,team,spread,date,position,team-code,rebounds,rebounds_5g,rebounds_10g,hit%,delta,delta_5g,delta_10g,opponent,rank,std_dev,std%,rpm
4,Total Rebounds,Pascal Siakam,IND,7.5,2025-1-16,PF,IND,7.390244,7.8,8.3,43.9,-0.109756,0.3,0.8,DET,5,2.836177,2.605706,0.224412
10,Total Rebounds,Amen Thompson,HOU,9.5,2025-1-16,SF,HOU,7.756757,12.0,10.2,29.7,-1.743243,2.5,0.7,SAC,18,3.435157,2.258050,0.259054
3,Total Rebounds,Myles Turner,IND,6.5,2025-1-16,C,IND,7.052632,6.6,7.0,52.6,0.552632,0.1,0.5,DET,2,2.459921,2.867016,0.227483
2,Total Rebounds,Jalen Duren,DET,9.5,2025-1-16,C,DET,9.378378,10.0,9.9,43.2,-0.121622,0.5,0.4,IND,22,4.198884,2.233540,0.371983
0,Total Rebounds,Ausar Thompson,DET,5.5,2025-1-16,SF,DET,4.666667,5.6,5.5,27.8,-0.833333,0.1,0.0,IND,11,2.634611,1.771292,0.241947
1,Total Rebounds,Cade Cunningham,DET,5.5,2025-1-16,PG,DET,6.555556,5.8,5.4,66.7,1.055556,0.3,-0.1,IND,3,2.568429,2.552360,0.185271
8,Total Rebounds,Jarrett Allen,CLE,10.5,2025-1-16,C,CLE,10.205128,11.2,10.4,48.7,-0.294872,0.7,-0.1,OKC,23,3.503228,2.913064,0.347602
7,Total Rebounds,Jalen Williams,OKC,5.5,2025-1-16,SG,OKC,5.650000,5.0,5.2,50.0,0.150000,-0.5,-0.3,CLE,15,2.213594,2.552410,0.180484
5,Total Rebounds,Tobias Harris,DET,6.5,2025-1-16,PF,DET,6.447368,5.4,6.1,55.3,-0.052632,-1.1,-0.4,IND,28,3.046463,2.116345,0.202065
9,Total Rebounds,Shai Gilgeous-Alexander,OKC,5.5,2025-1-16,PG,OKC,5.475000,5.8,4.9,45.0,-0.025000,0.3,-0.6,CLE,3,2.364128,2.315864,0.159708


# Assists

In [169]:
assists = props[props['type'] == 'Total Assists']

In [170]:
spread_merge = df[df['minutes'] != 0].merge(assists[['name','spread']],how='left')

In [171]:
spread_merge = spread_merge[~spread_merge['spread'].isna()]

In [172]:
spread_merge = spread_merge.groupby('name').apply(lambda group: (group['assists'] > group['spread']).mean() * 100).reset_index()

C:\Users\jtmck\AppData\Local\Temp\ipykernel_10048\1447438476.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  spread_merge = spread_merge.groupby('name').apply(lambda group: (group['assists'] > group['spread']).mean() * 100).reset_index()


In [173]:
spread_merge[0] = spread_merge[0].round(1)

In [174]:
spread_merge = spread_merge.rename(columns={0:'hit%'})

In [175]:
d1 = df.groupby(['name','position','team-code'])['assists'].mean().reset_index()

In [176]:
df5 = df[df['rank'] <= 5]
df10 = df[df['rank'] <= 10]

In [177]:
df5 = df5.groupby(['name'])['assists'].mean().reset_index()
df10 = df10.groupby(['name'])['assists'].mean().reset_index()

In [178]:
df5 = df5.rename(columns={'assists':'assists_5g'})
df10 = df10.rename(columns={'assists':'assists_10g'})

In [179]:
assists = assists.merge(d1,how='left').merge(df5,how='left').merge(df10,how='left').merge(spread_merge,how='left')

In [180]:
assists['delta'] = assists['assists'] - assists['spread']
assists['delta_5g'] = assists['assists_5g'] - assists['spread']
assists['delta_10g'] = assists['assists_10g'] - assists['spread']

In [181]:
assists = assists[~assists['assists'].isna()]

In [182]:
defense_assists = defense[defense['stat'] == 'assists']

In [183]:
defense_assists = defense_assists[['position','team','rank']]

In [184]:
defense_assists = defense_assists.rename(columns={'team':'opponent'})

In [185]:
assists.head()

,type,name,team,spread,date,position,team-code,assists,assists_5g,assists_10g,hit%,delta,delta_5g,delta_10g
0,Total Assists,Cade Cunningham,DET,9.5,2025-1-16,PG,DET,9.333333,8.6,8.4,47.2,-0.166667,-0.9,-1.1
1,Total Assists,Tyrese Haliburton,IND,9.5,2025-1-16,PG,IND,8.850000,8.8,9.0,40.0,-0.650000,-0.7,-0.5
2,Total Assists,Darius Garland,CLE,6.5,2025-1-16,PG,CLE,6.736842,6.4,7.8,50.0,0.236842,-0.1,1.3
3,Total Assists,Donovan Mitchell,CLE,4.5,2025-1-16,SG,CLE,4.486486,3.6,4.4,48.6,-0.013514,-0.9,-0.1
4,Total Assists,Jalen Williams,OKC,5.5,2025-1-16,SG,OKC,5.125000,5.4,5.7,37.5,-0.375000,-0.1,0.2


In [186]:
defense_assists

,position,opponent,rank
1149,PG,UTA,1
37,C,ORL,1
365,PF,SAS,1
181,PF,ORL,1
277,C,BOS,2
...,...,...,...
1133,PG,POR,26
285,PG,LAL,27
1173,PG,NOP,28
261,PG,MIA,28


In [187]:
assists['opponent'] = assists['team'].apply(lambda x: todays_games[x] if x in list(todays_games.keys()) else '')

In [188]:
assists = assists.merge(defense_assists,how='left')

In [189]:
assists.sort_values('delta_10g',ascending=False)

,type,name,team,spread,date,position,team-code,assists,assists_5g,assists_10g,hit%,delta,delta_5g,delta_10g,opponent,rank
2,Total Assists,Darius Garland,CLE,6.5,2025-1-16,PG,CLE,6.736842,6.4,7.8,50.0,0.236842,-0.1,1.3,OKC,4
4,Total Assists,Jalen Williams,OKC,5.5,2025-1-16,SG,OKC,5.125000,5.4,5.7,37.5,-0.375000,-0.1,0.2,CLE,6
5,Total Assists,Shai Gilgeous-Alexander,OKC,5.5,2025-1-16,PG,OKC,5.850000,5.0,5.5,52.5,0.350000,-0.5,0.0,CLE,16
3,Total Assists,Donovan Mitchell,CLE,4.5,2025-1-16,SG,CLE,4.486486,3.6,4.4,48.6,-0.013514,-0.9,-0.1,OKC,3
6,Total Assists,Fred VanVleet,HOU,6.5,2025-1-16,PG,HOU,6.081081,7.0,6.2,43.2,-0.418919,0.5,-0.3,SAC,7
1,Total Assists,Tyrese Haliburton,IND,9.5,2025-1-16,PG,IND,8.850000,8.8,9.0,40.0,-0.650000,-0.7,-0.5,DET,5
0,Total Assists,Cade Cunningham,DET,9.5,2025-1-16,PG,DET,9.333333,8.6,8.4,47.2,-0.166667,-0.9,-1.1,IND,14


In [190]:
dev_assists = df.groupby(['name'])['assists'].std().reset_index()

In [191]:
dev_assists = dev_assists.rename(columns={'assists':'std_dev'})

In [192]:
assists = assists.merge(dev_assists,how='left')

In [193]:
assists['std%'] = assists['assists']  / assists['std_dev']

In [194]:
df['apm'] = df['assists'] / df['minutes']

In [195]:
apm_df = df.groupby(['name'])['apm'].mean().reset_index()

In [196]:
assists = assists.merge(apm_df,how='left')

In [197]:
assists.sort_values('delta',ascending=False)

,type,name,team,spread,date,position,team-code,assists,assists_5g,assists_10g,hit%,delta,delta_5g,delta_10g,opponent,rank,std_dev,std%,apm
5,Total Assists,Shai Gilgeous-Alexander,OKC,5.5,2025-1-16,PG,OKC,5.850000,5.0,5.5,52.5,0.350000,-0.5,0.0,CLE,16,2.315500,2.526453,0.170763
2,Total Assists,Darius Garland,CLE,6.5,2025-1-16,PG,CLE,6.736842,6.4,7.8,50.0,0.236842,-0.1,1.3,OKC,4,2.468003,2.729673,0.223714
3,Total Assists,Donovan Mitchell,CLE,4.5,2025-1-16,SG,CLE,4.486486,3.6,4.4,48.6,-0.013514,-0.9,-0.1,OKC,3,2.022397,2.218400,0.143971
0,Total Assists,Cade Cunningham,DET,9.5,2025-1-16,PG,DET,9.333333,8.6,8.4,47.2,-0.166667,-0.9,-1.1,IND,14,3.295018,2.832559,0.262250
4,Total Assists,Jalen Williams,OKC,5.5,2025-1-16,SG,OKC,5.125000,5.4,5.7,37.5,-0.375000,-0.1,0.2,CLE,6,2.015167,2.543214,0.161923
6,Total Assists,Fred VanVleet,HOU,6.5,2025-1-16,PG,HOU,6.081081,7.0,6.2,43.2,-0.418919,0.5,-0.3,SAC,7,2.575293,2.361317,0.173710
1,Total Assists,Tyrese Haliburton,IND,9.5,2025-1-16,PG,IND,8.850000,8.8,9.0,40.0,-0.650000,-0.7,-0.5,DET,5,3.068032,2.884585,0.256817


# 3 Pointers

In [198]:
props['type'].unique()

array(['Total Points', 'Total Rebounds', 'Total Made 3 Points Shots',
       'Total Assists', 'Total Points, Rebounds and Assists'],
      dtype=object)

In [199]:
three_pointers = props[props['type'] == 'Total Made 3 Points Shots']

In [200]:
spread_merge = df[df['minutes'] != 0].merge(assists[['name','spread']],how='left')

In [201]:
df

,id,game-code,gameday,opponent,opponent-team-code,winorloss,result,statline,minutes,points,rebounds,assists,code,name,team,team-code,position,rank,pra,ppm,rpm,apm
4710,12155546,1081693,2025-01-15,Minnesota Timberwolves,MIN,W,"W, 116-115",19m 7p 6r 1a,19.0,7.0,6.0,1.0,124084,Gary Payton,Golden State,GSW,SG,1.0,14.0,0.368421,0.315789,0.052632
7301,12155569,1081698,2025-01-15,Los Angeles Lakers,LAL,L,"L, 108-117",22m 7p 2r 1a 1b,22.0,7.0,2.0,1.0,664841,Duncan Robinson,Miami,MIA,SF,1.0,10.0,0.318182,0.090909,0.045455
590,12155359,1081690,2025-01-15,Toronto Raptors,TOR,L,"L, 97-110",35m 18p 8r 3a 2b,35.0,18.0,8.0,3.0,3103,Kristaps Porzingis,Boston,BOS,C,1.0,29.0,0.514286,0.228571,0.085714
8691,12155334,1081689,2025-01-15,Philadelphia 76ers,PHI,W,"W, 125-119",45m 17p 4r 4a,45.0,17.0,4.0,4.0,329870,OG Anunoby,New York,NYK,PF,1.0,25.0,0.377778,0.088889,0.088889
3872,12155512,1081696,2025-01-15,Houston Rockets,HOU,L,"L, 108-128",25m 10p 1r 3a,25.0,10.0,1.0,3.0,52566239,Julian Strawther,Denver,DEN,SG,1.0,14.0,0.400000,0.040000,0.120000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8775,12099441,1080712,2024-10-22,Boston Celtics,BOS,L,"L, 109-132",13m 3p 1r,13.0,3.0,1.0,0.0,58076012,Pacome Dadiet,New York,NYK,SG,12.0,4.0,0.230769,0.076923,0.000000
1317,12099673,1080716,2024-10-22,Minnesota Timberwolves,MIN,W,"W, 110-103",34m 9p 1r 5a,34.0,9.0,1.0,5.0,3372,D'Angelo Russell,Brooklyn,BRK,PG,29.0,15.0,0.264706,0.029412,0.147059
9006,12099432,1080712,2024-10-22,Boston Celtics,BOS,L,"L, 109-132",24m 12p 7r 3a,24.0,12.0,7.0,3.0,3893,Karl-Anthony Towns,New York,NYK,C,38.0,22.0,0.500000,0.291667,0.125000
7908,12099662,1080716,2024-10-22,Los Angeles Lakers,LAL,L,"L, 103-110",32m 10p 3r 3a,32.0,10.0,3.0,3.0,665175,Donte DiVincenzo,Minnesota,MIN,SG,40.0,16.0,0.312500,0.093750,0.093750


In [202]:
spread_merge.head()

,id,game-code,gameday,opponent,opponent-team-code,winorloss,result,statline,minutes,points,rebounds,assists,code,name,team,team-code,position,rank,pra,ppm,rpm,apm,spread
0,12155546,1081693,2025-01-15,Minnesota Timberwolves,MIN,W,"W, 116-115",19m 7p 6r 1a,19.0,7.0,6.0,1.0,124084,Gary Payton,Golden State,GSW,SG,1.0,14.0,0.368421,0.315789,0.052632,NaN
1,12155569,1081698,2025-01-15,Los Angeles Lakers,LAL,L,"L, 108-117",22m 7p 2r 1a 1b,22.0,7.0,2.0,1.0,664841,Duncan Robinson,Miami,MIA,SF,1.0,10.0,0.318182,0.090909,0.045455,NaN
2,12155359,1081690,2025-01-15,Toronto Raptors,TOR,L,"L, 97-110",35m 18p 8r 3a 2b,35.0,18.0,8.0,3.0,3103,Kristaps Porzingis,Boston,BOS,C,1.0,29.0,0.514286,0.228571,0.085714,NaN
3,12155334,1081689,2025-01-15,Philadelphia 76ers,PHI,W,"W, 125-119",45m 17p 4r 4a,45.0,17.0,4.0,4.0,329870,OG Anunoby,New York,NYK,PF,1.0,25.0,0.377778,0.088889,0.088889,NaN
4,12155512,1081696,2025-01-15,Houston Rockets,HOU,L,"L, 108-128",25m 10p 1r 3a,25.0,10.0,1.0,3.0,52566239,Julian Strawther,Denver,DEN,SG,1.0,14.0,0.400000,0.040000,0.120000,NaN


In [203]:
spread_merge = spread_merge[~spread_merge['spread'].isna()]

In [204]:
spread_merge = spread_merge.groupby('name').apply(lambda group: (group['assists'] > group['spread']).mean() * 100).reset_index()

C:\Users\jtmck\AppData\Local\Temp\ipykernel_10048\1447438476.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  spread_merge = spread_merge.groupby('name').apply(lambda group: (group['assists'] > group['spread']).mean() * 100).reset_index()


In [205]:
spread_merge[0] = spread_merge[0].round(1)

In [206]:
spread_merge = spread_merge.rename(columns={0:'hit%'})

In [207]:
d1 = df.groupby(['name','position','team-code'])['assists'].mean().reset_index()

In [208]:
df5 = df[df['rank'] <= 5]
df10 = df[df['rank'] <= 10]

In [209]:
df5 = df5.groupby(['name'])['assists'].mean().reset_index()
df10 = df10.groupby(['name'])['assists'].mean().reset_index()

In [210]:
df5 = df5.rename(columns={'assists':'assists_5g'})
df10 = df10.rename(columns={'assists':'assists_10g'})

In [211]:
assists = assists.merge(d1,how='left').merge(df5,how='left').merge(df10,how='left').merge(spread_merge,how='left')

In [212]:
assists['delta'] = assists['assists'] - assists['spread']
assists['delta_5g'] = assists['assists_5g'] - assists['spread']
assists['delta_10g'] = assists['assists_10g'] - assists['spread']

In [213]:
assists = assists[~assists['assists'].isna()]

In [214]:
defense_assists = defense[defense['stat'] == 'assists']

In [215]:
defense_assists = defense_assists[['position','team','rank']]

In [216]:
defense_assists = defense_assists.rename(columns={'team':'opponent'})

In [217]:
assists.head()

,type,name,team,spread,date,position,team-code,assists,assists_5g,assists_10g,hit%,delta,delta_5g,delta_10g,opponent,rank,std_dev,std%,apm
0,Total Assists,Cade Cunningham,DET,9.5,2025-1-16,PG,DET,9.333333,8.6,8.4,47.2,-0.166667,-0.9,-1.1,IND,14,3.295018,2.832559,0.262250
1,Total Assists,Tyrese Haliburton,IND,9.5,2025-1-16,PG,IND,8.850000,8.8,9.0,40.0,-0.650000,-0.7,-0.5,DET,5,3.068032,2.884585,0.256817
2,Total Assists,Darius Garland,CLE,6.5,2025-1-16,PG,CLE,6.736842,6.4,7.8,50.0,0.236842,-0.1,1.3,OKC,4,2.468003,2.729673,0.223714
3,Total Assists,Donovan Mitchell,CLE,4.5,2025-1-16,SG,CLE,4.486486,3.6,4.4,48.6,-0.013514,-0.9,-0.1,OKC,3,2.022397,2.218400,0.143971
4,Total Assists,Jalen Williams,OKC,5.5,2025-1-16,SG,OKC,5.125000,5.4,5.7,37.5,-0.375000,-0.1,0.2,CLE,6,2.015167,2.543214,0.161923


In [218]:
defense_assists

,position,opponent,rank
1149,PG,UTA,1
37,C,ORL,1
365,PF,SAS,1
181,PF,ORL,1
277,C,BOS,2
...,...,...,...
1133,PG,POR,26
285,PG,LAL,27
1173,PG,NOP,28
261,PG,MIA,28


In [219]:
assists['opponent'] = assists['team'].apply(lambda x: todays_games[x] if x in list(todays_games.keys()) else '')

In [220]:
assists = assists.merge(defense_assists,how='left')

In [221]:
assists.sort_values('delta_10g',ascending=False)

,type,name,team,spread,date,position,team-code,assists,assists_5g,assists_10g,hit%,delta,delta_5g,delta_10g,opponent,rank,std_dev,std%,apm
2,Total Assists,Darius Garland,CLE,6.5,2025-1-16,PG,CLE,6.736842,6.4,7.8,50.0,0.236842,-0.1,1.3,OKC,4,2.468003,2.729673,0.223714
4,Total Assists,Jalen Williams,OKC,5.5,2025-1-16,SG,OKC,5.125000,5.4,5.7,37.5,-0.375000,-0.1,0.2,CLE,6,2.015167,2.543214,0.161923
5,Total Assists,Shai Gilgeous-Alexander,OKC,5.5,2025-1-16,PG,OKC,5.850000,5.0,5.5,52.5,0.350000,-0.5,0.0,CLE,16,2.315500,2.526453,0.170763
3,Total Assists,Donovan Mitchell,CLE,4.5,2025-1-16,SG,CLE,4.486486,3.6,4.4,48.6,-0.013514,-0.9,-0.1,OKC,3,2.022397,2.218400,0.143971
6,Total Assists,Fred VanVleet,HOU,6.5,2025-1-16,PG,HOU,6.081081,7.0,6.2,43.2,-0.418919,0.5,-0.3,SAC,7,2.575293,2.361317,0.173710
1,Total Assists,Tyrese Haliburton,IND,9.5,2025-1-16,PG,IND,8.850000,8.8,9.0,40.0,-0.650000,-0.7,-0.5,DET,5,3.068032,2.884585,0.256817
0,Total Assists,Cade Cunningham,DET,9.5,2025-1-16,PG,DET,9.333333,8.6,8.4,47.2,-0.166667,-0.9,-1.1,IND,14,3.295018,2.832559,0.262250


In [222]:
dev_assists = df.groupby(['name'])['assists'].std().reset_index()

In [223]:
dev_assists = dev_assists.rename(columns={'assists':'std_dev'})

In [224]:
assists = assists.merge(dev_assists,how='left')

In [225]:
assists['std%'] = assists['assists']  / assists['std_dev']

In [226]:
df['apm'] = df['assists'] / df['minutes']

In [227]:
apm_df = df.groupby(['name'])['apm'].mean().reset_index()

In [228]:
assists = assists.merge(apm_df,how='left')

In [229]:
assists.sort_values('delta',ascending=False)

,type,name,team,spread,date,position,team-code,assists,assists_5g,assists_10g,hit%,delta,delta_5g,delta_10g,opponent,rank,std_dev,std%,apm
5,Total Assists,Shai Gilgeous-Alexander,OKC,5.5,2025-1-16,PG,OKC,5.850000,5.0,5.5,52.5,0.350000,-0.5,0.0,CLE,16,2.315500,2.526453,0.170763
2,Total Assists,Darius Garland,CLE,6.5,2025-1-16,PG,CLE,6.736842,6.4,7.8,50.0,0.236842,-0.1,1.3,OKC,4,2.468003,2.729673,0.223714
3,Total Assists,Donovan Mitchell,CLE,4.5,2025-1-16,SG,CLE,4.486486,3.6,4.4,48.6,-0.013514,-0.9,-0.1,OKC,3,2.022397,2.218400,0.143971
0,Total Assists,Cade Cunningham,DET,9.5,2025-1-16,PG,DET,9.333333,8.6,8.4,47.2,-0.166667,-0.9,-1.1,IND,14,3.295018,2.832559,0.262250
4,Total Assists,Jalen Williams,OKC,5.5,2025-1-16,SG,OKC,5.125000,5.4,5.7,37.5,-0.375000,-0.1,0.2,CLE,6,2.015167,2.543214,0.161923
6,Total Assists,Fred VanVleet,HOU,6.5,2025-1-16,PG,HOU,6.081081,7.0,6.2,43.2,-0.418919,0.5,-0.3,SAC,7,2.575293,2.361317,0.173710
1,Total Assists,Tyrese Haliburton,IND,9.5,2025-1-16,PG,IND,8.850000,8.8,9.0,40.0,-0.650000,-0.7,-0.5,DET,5,3.068032,2.884585,0.256817


# 100% Hit Rate

In [ ]:
p1.groupby('name')[['points']].min().reset_index().rename(columns={'points':'points_min'})

# 90% Hit Rate

In [ ]:
df90 = df[df['minutes'] != 0]

In [ ]:
df90 = df90.groupby(['name','team'])[['points','rebounds','assists']].quantile(0.1).reset_index().sort_values('points',ascending=False)

In [ ]:
df90.head(100)

# 80% Hit Rate

In [ ]:
df80 = df[df['minutes'] != 0]

In [ ]:
df80 = df80.groupby(['name','team'])[['points','rebounds','assists']].quantile(0.2).reset_index().sort_values('points',ascending=False)

In [ ]:
df80.head(100)

# Draft Kings Lines

In [ ]:
url = "https://sportsbook.draftkings.com/leagues/basketball/nba?category=player-points&subcategory=points"

In [ ]:
options = webdriver.ChromeOptions()
#options.add_argument("--enable-javascript")
options.add_argument("javascript.enabled")
driver = webdriver.Chrome(options=options)
driver.get(url)
time.sleep(5)
#games = driver.find_element(By.CLASS_NAME,"game-view-cta")
html = driver.page_source
driver.quit()
soup = BeautifulSoup(html)

In [ ]:
rows = soup.find_all('div',{"class":"sb-market__template--2-columns-big-cells"})

In [ ]:
points_lines = []
for row in rows:
    name = row.find('ul').find('a').text
    line = int(row.find('button', {'class':"sb-selection-picker__selection sb-selection-picker__selection--focused"}).text.split('+')[0]) - 0.5
    points_lines.append({'name':name,'spread':line})

In [ ]:
points_props = pd.DataFrame(points_lines)

In [ ]:
points_props.head()

In [ ]:
points_props.shape

In [ ]:
players1 = players[['name','team-code']]

In [ ]:
players1 = players1.rename(columns={'team-code':'team'})

In [ ]:
players1[players1['name'].str.contains('Jaime')]

In [ ]:
points_props.merge(players1,how='left')

In [ ]:
d1 = df.copy()

In [293]:
spread_merge = df[df['minutes'] != 0].merge(points_props[['name','spread']],how='left')

In [294]:
spread_merge = spread_merge[~spread_merge['spread'].isna()]

In [295]:
spread_merge = spread_merge.groupby('name').apply(lambda group: (group['points'] > group['spread']).mean() * 100).reset_index()

C:\Users\jtmck\AppData\Local\Temp\ipykernel_83572\2933933359.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  spread_merge = spread_merge.groupby('name').apply(lambda group: (group['points'] > group['spread']).mean() * 100).reset_index()


In [296]:
spread_merge[0] = spread_merge[0].round(1)

In [297]:
spread_merge = spread_merge.rename(columns={0:'hit%'})

In [298]:
d1['pra'] = d1['points'] + d1['rebounds'] + d1['assists']
df['pra'] = df['points'] + df['rebounds'] + df['assists']

In [299]:
d2 = d1.groupby(['name','team-code','position'])['points'].mean().reset_index()

In [300]:
df5 = d1[d1['rank'] <= 5]

In [301]:
df10 = d1[d1['rank'] <= 10]

In [302]:
df5 = df5.groupby(['name','team-code','position'])['points'].mean().reset_index()
df10 = df10.groupby(['name','team-code','position'])['points'].mean().reset_index()

In [303]:
df5 = df5.rename(columns={'points':'points_5g'})
df10 = df10.rename(columns={'points':'points_10g'})

In [304]:
p1 = df[df['minutes'] != 0]

In [305]:
points_props = points_props.merge(d2,how='left').merge(df5,how='left')
points_props = points_props.merge(df10,how='left').merge(spread_merge,how='left')
points_props = points_props.merge(p1.groupby('name')[['points']].min().reset_index().rename(columns={'points':'points_min'}),how='left')

In [306]:
points_props.head()

,type,name,team,spread,date,team-code,position,points,points_5g,points_10g,hit%,points_min
0,Total Points,Cam Thomas,BKN,19.5,2024-12-29,BRK,SG,24.705882,26.8,23.4,58.8,13.0
1,Total Points,Cameron Johnson,BKN,18.5,2024-12-29,BRK,PF,19.448276,24.2,22.4,48.3,5.0
2,Total Points,Dorian Finney-Smith,BKN,8.5,2024-12-29,BRK,PF,10.400000,9.6,10.8,65.0,2.0
3,Total Points,Goga Bitadze,ORL,12.5,2024-12-29,ORL,C,9.607143,13.0,12.1,17.9,0.0
4,Total Points,Jalen Suggs,ORL,22.5,2024-12-29,ORL,PG,16.806452,18.0,20.1,25.8,2.0


In [307]:
points_props['delta'] = points_props['points'] - points_props['spread']

In [308]:
points_props['delta_5g'] = points_props['points_5g'] - points_props['spread']
points_props['delta_10g'] = points_props['points_10g'] - points_props['spread']

In [309]:
points_props = points_props[~points_props['delta'].isna()]

In [310]:
points_props['team-code'].unique()

array(['BRK', 'ORL', 'IND', 'BOS', 'ATL', 'TOR', 'MEM', 'OKC', 'HOU',
       'MIA', 'MIN', 'SAS'], dtype=object)

In [311]:
points_props[points_props['team-code'] == 'MIL']

,type,name,team,spread,date,team-code,position,points,points_5g,points_10g,hit%,points_min,delta,delta_5g,delta_10g


In [312]:
points_props = points_props[~points_props['team-code'].isin(['MIL','OKC'])]

In [313]:
points_props['opponent'] = points_props['team-code'].apply(lambda x: todays_games[x])

In [314]:
defense['team'] = defense['team'].apply(lambda x: 'GSW' if x == 'GS' else 'BRK' if x == 'BKN' else 'SAS' if x == 'SA' else 'NOP' if x == 'NO' else 'NYK' if x == 'NY' else 'CHH' if x == 'CHA' else x)

In [315]:
defense_points = defense[defense['stat'] == 'points']

In [316]:
defense_points = defense_points[['position','team','rank']]

In [317]:
defense_points = defense_points.rename(columns={'team':'opponent'})

In [318]:
points_props = points_props.merge(defense_points,how='left')

In [319]:
dev_points = df.groupby(['name'])['points'].std().reset_index()

In [320]:
dev_points = dev_points.rename(columns={'points':'std_dev'})

In [321]:
points_props = points_props.merge(dev_points,how='left')

In [322]:
points_props['std%'] = points_props['std_dev'] / points_props['points']

In [323]:
df['ppm'] = df['points'] / df['minutes']

In [324]:
ppm_df = df.groupby(['name'])['ppm'].mean().reset_index()

In [325]:
points_props = points_props.merge(ppm_df,how='left')

In [326]:
pd.options.display.max_rows = 100

In [327]:
points_props.shape

(50, 20)

In [328]:
defense.team.unique()

array(['MIL', 'SAS', 'MIN', 'CLE', 'DET', 'LAL', 'ORL', 'NYK', 'SAC',
       'DAL', 'NOP', 'DEN', 'ATL', 'TOR', 'PHI', 'WAS', 'MEM', 'MIA',
       'CHH', 'PHO', 'LAC', 'CHI', 'OKC', 'POR', 'BOS', 'BRK', 'IND',
       'HOU', 'GSW', 'UTA'], dtype=object)

In [329]:
points_props.sort_values('delta_5g',ascending=False).head(100)

,type,name,team,spread,date,team-code,position,points,points_5g,points_10g,hit%,points_min,delta,delta_5g,delta_10g,opponent,rank,std_dev,std%,ppm
0,Total Points,Cam Thomas,BKN,19.5,2024-12-29,BRK,SG,24.705882,26.8,23.4,58.8,13.0,5.205882,7.3,3.9,ORL,15,8.571499,0.346942,0.737093
49,Total Points,Victor Wembanyama,SAS,24.5,2024-12-29,SAS,C,25.192308,31.8,28.1,53.8,6.0,0.692308,7.3,3.6,MIN,14,10.143054,0.402625,0.758311
30,Total Points,Dillon Brooks,HOU,12.5,2024-12-29,HOU,SF,13.642857,19.0,15.6,64.3,0.0,1.142857,6.5,3.1,MIA,15,7.155491,0.524486,0.431587
1,Total Points,Cameron Johnson,BKN,18.5,2024-12-29,BRK,PF,19.448276,24.2,22.4,48.3,5.0,0.948276,5.7,3.9,ORL,3,7.753277,0.398661,0.583551
6,Total Points,Andrew Nembhard,IND,11.5,2024-12-29,IND,SG,10.588235,17.2,12.9,41.2,2.0,-0.911765,5.7,1.4,BOS,8,5.863647,0.553789,0.418725
39,Total Points,Donte DiVincenzo,MIN,9.5,2024-12-29,MIN,SG,9.400000,15.0,10.1,46.7,2.0,-0.100000,5.5,0.6,SAS,16,5.256195,0.559170,0.364462
10,Total Points,Jayson Tatum,BOS,27.5,2024-12-29,BOS,PF,28.785714,31.2,29.3,64.3,15.0,1.285714,3.7,1.8,IND,28,7.425467,0.257957,0.795836
9,Total Points,Jaylen Brown,BOS,24.5,2024-12-29,BOS,SF,24.615385,28.0,24.0,50.0,11.0,0.115385,3.5,-0.5,IND,19,7.440844,0.302284,0.690189
33,Total Points,Jalen Green,HOU,18.5,2024-12-29,HOU,SG,19.322581,21.2,20.3,45.2,6.0,0.822581,2.7,1.8,MIA,8,9.162922,0.474208,0.587355
20,Total Points,Ochai Agbaji,TOR,10.5,2024-12-29,TOR,SG,11.548387,12.8,10.6,54.8,0.0,1.048387,2.3,0.1,ATL,23,5.971257,0.517064,0.369481
